# UCS Trailer: Nairobi to Kisumu

**ICS 3202: Artificial Intelligence — Practical Exercise 3B**

This notebook applies Uniform Cost Search (UCS) to the directed highway graph and step costs defined in Part 1. The search finds the least-distance route for the trailer from Nairobi to Kisumu.

**Group members:** Austine Igunza (166990), Paul Kiragu (190135), Mwijukye Chief Cedrick (192468).

> Run the cells from top to bottom. The graph and costs below are the group's Part 1 inputs; distances are treated as fixed edge costs for this search exercise.


## Step 1 — Goal formulation

- **Agent:** self-driving trailer
- **Initial state:** Nairobi
- **Goal state:** Kisumu
- **Objective:** minimize total road distance, measured in kilometres

## Step 2 — Problem formulation

A state is the town where the trailer is located. An action is travelling along one outgoing highway link. The path-cost function is `g(n) = sum of edge distances on the path`. The goal test is whether the current town is Kisumu.

The directed graph and costs are the Part 1 abstraction (towns are states; highway trips are actions):

![Part 1 directed highway graph](image.png)

The graph is represented in the next cell as an adjacency list. Each edge stores its destination, distance in kilometres, and highway label.


In [1]:
from heapq import heappop, heappush
from itertools import count
from math import inf

# GRAPH[town] = [(destination, distance_km, highway), ...]
GRAPH = {
    "Nairobi":   [("Thika", 45, "A2"), ("Naivasha", 90, "A104"), ("Narok", 142, "B3")],
    "Thika":     [("Nyeri", 106, "A2")],
    "Nyeri":     [("Nyahururu", 104, "B5")],
    "Nyahururu": [("Nakuru", 64, "B5")],
    "Naivasha":  [("Nakuru", 71, "A104"), ("Narok", 90, "B3")],
    "Narok":     [("Bomet", 97, "B3")],
    "Nakuru":    [("Kericho", 110, "B1"), ("Eldoret", 155, "A104")],
    "Bomet":     [("Kericho", 50, "C23"), ("Kisii", 68, "B3")],
    "Kericho":   [("Kisumu", 90, "B1"), ("Kisii", 85, "C23")],
    "Eldoret":   [("Kakamega", 80, "A104/B2")],
    "Kisii":     [("Kisumu", 118, "B1")],
    "Kakamega":  [("Kisumu", 52, "B2")],
    "Kisumu":    [],
}

START = "Nairobi"
GOAL = "Kisumu"

print(f"States (towns): {len(GRAPH)}")
print(f"Directed actions (highway links): {sum(len(edges) for edges in GRAPH.values())}")
print("\nFrom         To           Highway    Cost (km)")
print("-" * 49)
for town, edges in GRAPH.items():
    for destination, distance, highway in edges:
        print(f"{town:<13}{destination:<13}{highway:<11}{distance:>5}")


States (towns): 13
Directed actions (highway links): 18

From         To           Highway    Cost (km)
-------------------------------------------------
Nairobi      Thika        A2            45
Nairobi      Naivasha     A104          90
Nairobi      Narok        B3           142
Thika        Nyeri        A2           106
Nyeri        Nyahururu    B5           104
Nyahururu    Nakuru       B5            64
Naivasha     Nakuru       A104          71
Naivasha     Narok        B3            90
Narok        Bomet        B3            97
Nakuru       Kericho      B1           110
Nakuru       Eldoret      A104         155
Bomet        Kericho      C23           50
Bomet        Kisii        B3            68
Kericho      Kisumu       B1            90
Kericho      Kisii        C23           85
Eldoret      Kakamega     A104/B2       80
Kisii        Kisumu       B1           118
Kakamega     Kisumu       B2            52


## Step 3 — Uniform Cost Search

UCS stores candidate paths in a priority queue ordered by cumulative cost `g(n)`. It always expands the lowest-cost frontier node next. The goal test is applied when Kisumu is removed from the queue, which ensures the returned goal path has minimum cost when all edge costs are non-negative. A cheaper route to a town replaces its earlier cost; stale, more expensive queue entries are skipped.

The trace below shows each expanded town, its cumulative cost, the frontier ordered by cost, and any more expensive alternative discarded.


In [2]:
def uniform_cost_search(graph, start, goal):
    # Each queue item is (cumulative_cost, tie_breaker, town).
    tie_breaker = count()
    frontier = [(0, next(tie_breaker), start)]
    best_cost = {start: 0}
    parent = {}
    trace = []
    expanded = []

    while frontier:
        cost, _, town = heappop(frontier)
        if cost != best_cost.get(town):  # skip a stale, more expensive entry
            continue

        expanded.append((town, cost))
        if town == goal:  # test the goal when it is removed at minimum cost
            path = [goal]
            while path[-1] != start:
                path.append(parent[path[-1]][0])
            path.reverse()
            return path, cost, trace, expanded, parent

        discarded = []
        for destination, step_cost, highway in graph[town]:
            new_cost = cost + step_cost
            if new_cost < best_cost.get(destination, inf):
                best_cost[destination] = new_cost
                parent[destination] = (town, step_cost, highway)
                heappush(frontier, (new_cost, next(tie_breaker), destination))
            else:
                discarded.append(f"{destination} {new_cost} (existing best: {best_cost[destination]})")

        current_frontier = sorted(
            (queued_cost, queued_town)
            for queued_cost, _, queued_town in frontier
            if queued_cost == best_cost.get(queued_town)
        )
        trace.append({"town": town, "cost": cost,
                      "frontier": current_frontier, "discarded": discarded})

    raise ValueError(f"No route found from {start} to {goal}")

path, total_cost, search_trace, expanded, parent = uniform_cost_search(GRAPH, START, GOAL)

for step_number, row in enumerate(search_trace, start=1):
    frontier_text = ", ".join(f"{town} {cost}" for cost, town in row["frontier"])
    print(f"Step {step_number}: EXPAND {row['town']} (g = {row['cost']} km)")
    print(f"  Frontier: {frontier_text or 'empty'}")
    if row["discarded"]:
        print("  Discarded: " + "; ".join(row["discarded"]))
print(f"Step {len(search_trace) + 1}: EXPAND {GOAL} (g = {total_cost} km) — GOAL TEST PASSED")
print(f"Search terminates after {len(expanded)} expansions.")


Step 1: EXPAND Nairobi (g = 0 km)
  Frontier: Thika 45, Naivasha 90, Narok 142
Step 2: EXPAND Thika (g = 45 km)
  Frontier: Naivasha 90, Narok 142, Nyeri 151
Step 3: EXPAND Naivasha (g = 90 km)
  Frontier: Narok 142, Nyeri 151, Nakuru 161
  Discarded: Narok 180 (existing best: 142)
Step 4: EXPAND Narok (g = 142 km)
  Frontier: Nyeri 151, Nakuru 161, Bomet 239
Step 5: EXPAND Nyeri (g = 151 km)
  Frontier: Nakuru 161, Bomet 239, Nyahururu 255
Step 6: EXPAND Nakuru (g = 161 km)
  Frontier: Bomet 239, Nyahururu 255, Kericho 271, Eldoret 316
Step 7: EXPAND Bomet (g = 239 km)
  Frontier: Nyahururu 255, Kericho 271, Kisii 307, Eldoret 316
  Discarded: Kericho 289 (existing best: 271)
Step 8: EXPAND Nyahururu (g = 255 km)
  Frontier: Kericho 271, Kisii 307, Eldoret 316
  Discarded: Nakuru 319 (existing best: 161)
Step 9: EXPAND Kericho (g = 271 km)
  Frontier: Kisii 307, Eldoret 316, Kisumu 361
  Discarded: Kisii 356 (existing best: 307)
Step 10: EXPAND Kisii (g = 307 km)
  Frontier: Eldoret 3

### Search tree view

This Part 1 tree shows the cumulative path costs and the order of expansion. The highlighted route is the solution found by UCS.

![Uniform-cost search tree](image-1.png)


## Step 4 — Execute the solution

Reconstruct the actions from the parent links and add each step cost to show the cumulative path cost.


In [3]:
print(f"{'#':<4}{'Action':<43}{'Step cost (km)':>16}{'g(n) (km)':>13}")
print("-" * 76)
cumulative = 0
for index in range(1, len(path)):
    previous_town = path[index - 1]
    town = path[index]
    parent_town, step_cost, highway = parent[town]
    assert parent_town == previous_town
    cumulative += step_cost
    action = f"{previous_town} -> {town} via {highway}"
    print(f"{index:<4}{action:<43}{step_cost:>16}{cumulative:>13}")

print("-" * 76)
print("OPTIMAL PATH:", " -> ".join(path))
print(f"TOTAL PATH COST: {total_cost} km")


#   Action                                       Step cost (km)    g(n) (km)
----------------------------------------------------------------------------
1   Nairobi -> Naivasha via A104                             90           90
2   Naivasha -> Nakuru via A104                              71          161
3   Nakuru -> Kericho via B1                                110          271
4   Kericho -> Kisumu via B1                                 90          361
----------------------------------------------------------------------------
OPTIMAL PATH: Nairobi -> Naivasha -> Nakuru -> Kericho -> Kisumu
TOTAL PATH COST: 361 km


## Check against all simple routes

As a cross-check, enumerate every simple route in this finite graph and compare its total distance with UCS. This is an additional verification of the result, not the search strategy used to find it.


In [4]:
def enumerate_simple_routes(graph, start, goal):
    routes = []
    def visit(town, visited, towns, cost):
        if town == goal:
            routes.append((cost, towns.copy()))
            return
        for destination, step_cost, _highway in graph[town]:
            if destination not in visited:
                visit(destination, visited | {destination}, towns + [destination], cost + step_cost)
    visit(start, {start}, [start], 0)
    return sorted(routes)

routes = enumerate_simple_routes(GRAPH, START, GOAL)
for index, (cost, towns) in enumerate(routes, start=1):
    marker = "  <- UCS OPTIMUM" if towns == path else ""
    print(f"{index:>2}. {' -> '.join(towns)} = {cost} km{marker}")

assert routes[0][0] == total_cost
assert routes[0][1] == path
print(f"\nRoutes checked: {len(routes)}")
print(f"UCS agrees with exhaustive route comparison: {routes[0][0] == total_cost}")


 1. Nairobi -> Naivasha -> Nakuru -> Kericho -> Kisumu = 361 km  <- UCS OPTIMUM
 2. Nairobi -> Narok -> Bomet -> Kericho -> Kisumu = 379 km
 3. Nairobi -> Naivasha -> Narok -> Bomet -> Kericho -> Kisumu = 417 km
 4. Nairobi -> Narok -> Bomet -> Kisii -> Kisumu = 425 km
 5. Nairobi -> Naivasha -> Nakuru -> Eldoret -> Kakamega -> Kisumu = 448 km
 6. Nairobi -> Naivasha -> Narok -> Bomet -> Kisii -> Kisumu = 463 km
 7. Nairobi -> Naivasha -> Nakuru -> Kericho -> Kisii -> Kisumu = 474 km
 8. Nairobi -> Narok -> Bomet -> Kericho -> Kisii -> Kisumu = 492 km
 9. Nairobi -> Thika -> Nyeri -> Nyahururu -> Nakuru -> Kericho -> Kisumu = 519 km
10. Nairobi -> Naivasha -> Narok -> Bomet -> Kericho -> Kisii -> Kisumu = 530 km
11. Nairobi -> Thika -> Nyeri -> Nyahururu -> Nakuru -> Eldoret -> Kakamega -> Kisumu = 606 km
12. Nairobi -> Thika -> Nyeri -> Nyahururu -> Nakuru -> Kericho -> Kisii -> Kisumu = 632 km

Routes checked: 12
UCS agrees with exhaustive route comparison: True


## Result and assumptions

**Optimal route:** Nairobi → Naivasha → Nakuru → Kericho → Kisumu  
**Total path cost:** 90 + 71 + 110 + 90 = **361 km**

The result is optimal for the directed graph and edge costs used here. The model treats the Part 1 distances as fixed and does not include traffic, road closures, fuel, travel time, or changing road conditions. The group’s Part 1 search-tree diagram is also available as `image-1.png` in this repository.


## Evidence of groupwork

The photograph below records the group working together on the exercise. The image is also included as a separate repository file so it remains available when the notebook is opened from GitHub.

![Groupwork evidence](groupwork_exercise_3b.jpeg)
